# Jetson Orin Nano 모델 선정 보고서: YOLOv8n + 웃음 감지

- 작성 기준일: 2026-07-21
- 대상 장치: NVIDIA Jetson Orin Nano
- 영상 요구사항: 사람 탐지(`person`)
- 오디오 요구사항: 가족 대화 환경에서 웃음 이벤트 감지

## 최종 선택 한눈에 보기

| 기능 | 최종 모델 | 실행 위치 | 배포 형식 | 결정 |
|---|---|---|---|---|
| 사람 탐지 | **YOLOv8n** | **Jetson Orin Nano GPU** | **TensorRT FP16** | 사용 |
| 웃음·생활음 감지 | **YAMNet** | **Raspberry Pi 5 CPU** 권장, 단독 구성은 Jetson CPU | **TFLite** | 사용 |
| 오디오 대체 후보 | EfficientAT MN04 | 필요 시 재평가 | PyTorch | 현재 미사용 |
| 고성능 오디오 후보 | EfficientAT MN10 | - | PyTorch | 사용하지 않음 |

따라서 두 보드를 함께 쓸 때는 **Jetson = YOLOv8n**, **Pi 5 = YAMNet TFLite**로 역할을 나눈다. Jetson 한 대만 사용할 때는 YOLOv8n을 GPU/TensorRT로, YAMNet을 CPU/TFLite로 동시에 실행한다.

> 주의: COCO mAP와 AudioSet mAP는 서로 다른 과제의 지표이므로 직접 비교하지 않는다. AudioSet 전체 mAP 역시 웃음 클래스의 정밀도·재현율을 대신하지 않는다. 최종 임계값은 반드시 현장 녹음으로 정한다.

## 1. 모델 개발·배포 주체

| 모델 | 개발 주체 | 기반 기술·학습 데이터 | 공식 배포처 | 라이선스 메모 |
|---|---|---|---|---|
| YOLOv8n | **Ultralytics** | PyTorch 기반 YOLOv8, COCO 80개 클래스 사전학습 | Ultralytics 문서·GitHub Assets | AGPL-3.0 또는 Enterprise. 비공개·상용·임베디드 제품은 출시 전 조건 확인 필수 |
| YAMNet | **Google Sound Understanding / TensorFlow 팀** | MobileNet v1 기반, Google AudioSet의 521개 음향 클래스 | TensorFlow Hub·Google TFLite 배포 | 공식 모델 카드와 TensorFlow 배포 조건 확인 |
| EfficientAT MN04/MN10 | **Johannes Kepler University Linz, Institute of Computational Perception**의 Florian Schmid·Khaled Koutini·Gerhard Widmer 연구팀 | MobileNetV3 기반, AudioSet 527개 클래스, Transformer-to-CNN 지식 증류 | 연구팀 공식 GitHub Releases | 코드·체크포인트 저장소 MIT License |

YOLOv8n과 YAMNet은 기업이 관리하는 범용 배포 모델이고, EfficientAT은 ICASSP 2023에 발표된 대학 연구 모델이다. 이 프로젝트에서는 성능뿐 아니라 Jetson/Pi 배포 난이도와 런타임 의존성까지 포함해 선택한다.

## 2. 공식 자료 기반 후보 비교

### YOLOv8n

Ultralytics 공식 표의 640 입력 기준 수치다. A100/TensorRT 또는 일반 CPU 수치는 Jetson 실측값이 아니므로 모델 간 참고치로만 쓴다.

| 모델 | 파라미터 | FLOPs | COCO mAP50-95 | 공식 CPU ONNX | 체크포인트 |
|---|---:|---:|---:|---:|---:|
| YOLOv8n | 3.2M | 8.7B | 37.3 | 80.4 ms | 이 저장소 실측 약 6.25 MiB |

### 웃음 감지 후보

| 모델 | 구조/런타임 | 입력 | 전체 AudioSet 성능 | 다운로드 파일 | 판단 |
|---|---|---|---:|---:|---|
| YAMNet TFLite | MobileNet v1 / TFLite | 16 kHz mono | 521개 클래스; 모델 카드에서 클래스별 검증 필요 | 약 3.94 MiB | MVP 권고 |
| EfficientAT MN04 | MobileNetV3 / PyTorch | 32 kHz mono | mAP 0.432 | 약 3.88 MiB | 정확도 challenger |
| EfficientAT MN10 | MobileNetV3 / PyTorch | 32 kHz mono | mAP 0.471 | 약 18.80 MiB | 보류 |

EfficientAT 수치는 공식 저장소의 AudioSet 평가 결과다. YAMNet은 521개 AudioSet 이벤트를 분류하며 `Laughter` 계열을 포함한다. 두 구현의 클래스 수와 전처리가 다르므로 점수 임계값을 그대로 공유하면 안 된다.

In [1]:
from pathlib import Path
import hashlib, json, os, platform, statistics, subprocess, sys, time
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'model_benchmark':
    candidate = ROOT / 'ai' / 'model_benchmark'
    ROOT = candidate if candidate.exists() else ROOT
MODELS = ROOT / 'models'
DATA = ROOT / 'data'
CACHE = ROOT / 'cache'
for path in (MODELS, DATA, CACHE): path.mkdir(parents=True, exist_ok=True)
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'root': str(ROOT.resolve())})

{'python': '3.12.12', 'platform': 'Windows-11-10.0.26200-SP0', 'root': 'C:\\Users\\SSAFY\\cksgud\\higher\\S15P11E201\\ai\\model_benchmark'}


## 3. 다운로드한 모델 무결성·용량 확인

모델은 공식 배포 주소에서 내려받았다. 셀을 다시 실행해 누락 파일만 받을 수 있다. 바이너리는 재현성을 위해 SHA-256을 함께 기록한다.

In [2]:
from urllib.request import urlretrieve

MODEL_URLS = {
    'yolov8n.pt': 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt',
    'yamnet.tflite': 'https://storage.googleapis.com/download.tensorflow.org/models/tflite/task_library/audio_classification/android/lite-model_yamnet_classification_tflite_1.tflite',
    'efficientat_mn04_as.pt': 'https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/mn04_as_mAP_432.pt',
    'efficientat_mn10_as.pt': 'https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/mn10_as_mAP_471.pt',
}
for name, url in MODEL_URLS.items():
    target = MODELS / name
    if not target.exists():
        print('download:', name)
        urlretrieve(url, target)

rows = []
for name in MODEL_URLS:
    path = MODELS / name
    rows.append({
        'file': name,
        'size_MiB': round(path.stat().st_size / 1024**2, 2),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    })
manifest = pd.DataFrame(rows)
manifest

,file,size_MiB,sha256
0,yolov8n.pt,6.25,f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d...
1,yamnet.tflite,3.94,10c95ea3eb9a7bb4cb8bddf6feb023250381008177ac16...
2,efficientat_mn04_as.pt,3.88,899a8c6217063f6941793102a2780b3dd788ab11fce468...
3,efficientat_mn10_as.pt,18.80,0bd7dc2443af498c289a2e739f02ebb515d6aa3fd3ab9d...


## 4. YOLOv8n 로컬 스모크 테스트

아래 수치는 **현재 노트북을 실행한 장치**의 결과다. Jetson 결과가 아니면 최종 성능표에 섞지 않는다. 실제 배포는 Jetson에서 TensorRT 엔진을 생성해야 하며, 다른 장치에서 만든 `.engine` 파일을 복사해 쓰지 않는다.

필요 패키지: `pip install ultralytics onnx onnxruntime`

In [3]:
# 인터넷 연결 시 공식 예제 이미지를 한 번만 받는다.
BUS = DATA / 'bus.jpg'
if not BUS.exists():
    urlretrieve('https://ultralytics.com/images/bus.jpg', BUS)

try:
    import torch
    from ultralytics import YOLO
    yolo = YOLO(str(MODELS / 'yolov8n.pt'))
    device = 0 if torch.cuda.is_available() else 'cpu'
    for _ in range(3): yolo.predict(str(BUS), imgsz=640, classes=[0], device=device, verbose=False)
    elapsed = []
    person_counts = []
    for _ in range(20):
        t0 = time.perf_counter()
        result = yolo.predict(str(BUS), imgsz=640, classes=[0], device=device, verbose=False)[0]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed.append((time.perf_counter() - t0) * 1000)
        person_counts.append(len(result.boxes))
    yolo_result = {
        'device': str(device),
        'hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor(),
        'median_ms_end_to_end': round(statistics.median(elapsed), 2),
        'p95_ms': round(float(np.percentile(elapsed, 95)), 2), 'person_count': person_counts[-1],
    }
except Exception as exc:
    yolo_result = {'status': 'SKIPPED', 'reason': repr(exc)}
yolo_result

{'device': '0',
 'hardware': 'NVIDIA GeForce RTX 4070 Laptop GPU',
 'median_ms_end_to_end': 15.78,
 'p95_ms': 22.15,
 'person_count': 4}

In [4]:
# Jetson에서만 실행 권장: TensorRT FP16 엔진 생성 및 측정
# engine_path = yolo.export(format='engine', imgsz=640, half=True, dynamic=False, workspace=2)
# trt = YOLO(engine_path)
# 위와 같은 3회 warm-up + 20회 측정 루프를 trt.predict(..., classes=[0])에 적용한다.

## 5. 웃음 데이터셋과 평가 규칙

모델 선택은 파일 크기나 AudioSet 전체 mAP만으로 끝내면 안 된다. 최소 100개 이상의 현장 클립을 다음처럼 구성한다.

- Positive: 성인 웃음, 아이 웃음, 작은 킥킥거림, 여러 명 동시 웃음
- Hard negative: 평범한 대화, TV 예능 웃음, 울음, 기침, 재채기, 고성, 음악, 식기·청소기 소음
- 권장 비율: positive 40%, hard negative 60%; 화자와 녹음 세션 단위로 train/validation/test 분리
- 주 지표: 이벤트 단위 Precision, Recall, F1, 시간당 오탐 수(FP/hour), 검출 지연
- 초기 합격선 제안: Recall ≥ 0.90, Precision ≥ 0.85, FP/hour ≤ 1.0

한국인 스마트폰 녹음 기반 공개 샘플로는 Deeply Nonverbal Vocalization Dataset(OpenSLR 99)을 참고할 수 있다. 라이선스는 CC BY-NC-ND 4.0이므로 연구 비교용으로만 사용하고, 제품 학습 데이터로 재배포하거나 변형하기 전 조건을 다시 확인한다. 현장 최종 평가는 팀이 직접 수집하고 동의받은 데이터로 수행한다.

### 공개 한국어 샘플 탐색 결과(실측)

OpenSLR 99에서 웃음 25개와 나머지 15종 비언어 발성별 5개씩 총 75개를 negative로 사용했다. `Laughter`, `Baby laughter`, `Giggle`, `Snicker`, `Belly laugh` 점수 중 최대값을 이벤트 점수로 사용했다. YAMNet은 50% 겹침 창의 최대 점수를, EfficientAT은 전체 짧은 클립 점수를 사용했다.

| 모델 | AUPRC | AUROC | 탐색 최적 F1 | Precision | Recall | CPU median | CPU p95 |
|---|---:|---:|---:|---:|---:|---:|---:|
| YAMNet TFLite | 0.9985 | 0.9997 | 0.9804 | 0.9615 | 1.0000 | 4.85 ms | 10.62 ms |
| EfficientAT MN04 | 1.0000 | 1.0000 | 1.0000 | 1.0000 | 1.0000 | 19.78 ms | 25.35 ms |
| EfficientAT MN10 | 0.2167 | 0.4432 | 0.4717 | 0.3086 | 1.0000 | 26.41 ms | 34.29 ms |

측정 장치 CPU는 Intel64 Family 6 Model 170이며 모델 전처리를 포함했다. 임계값을 같은 100개 샘플에서 선택했으므로 F1·Precision·Recall은 낙관적인 **탐색 결과**이고 독립 테스트 성능이 아니다. AUPRC/AUROC도 작은 공개 샘플에 한정된다. 특히 MN10의 낮은 결과는 공식 AudioSet 전체 mAP와 상반되므로 배포 부적합 신호로만 보고, 구현·짧은 클립 길이·도메인 차이를 별도 조사해야 한다.

MN04가 이 표에서는 완벽하게 분리했지만 YAMNet 대비 F1 차이는 약 1.96%p이고 중앙 지연시간은 약 4.1배다. 따라서 현재 자료만으로는 5%p 교체 기준을 충족하지 않아 **YAMNet TFLite 권고를 유지**한다.

In [5]:
# 선택 실행: 약 0.1 GB의 한국어 비언어 발성 샘플 전체 다운로드
# archive = DATA / 'NonverbalVocalization.tgz'
# if not archive.exists():
#     urlretrieve('https://www.openslr.org/resources/99/NonverbalVocalization.tgz', archive)
# subprocess.run(['tar', '-xzf', str(archive), '-C', str(DATA)], check=True)

## 6. YAMNet TFLite 지연시간·점수 확인

`data/eval/laughter/*.wav`와 `data/eval/non_laughter/*.wav`에 16-bit WAV 파일을 넣는다. 모델 메타데이터의 입력 길이에 맞춰 중앙 구간을 자르거나 0으로 패딩한다. 아래 셀은 런타임 지연시간을 측정하고 raw score를 반환한다. 실제 이벤트 판정은 한 프레임 임계값보다 연속 프레임·히스테리시스가 안전하다.

Jetson에서는 `tflite-runtime`이 제공되는 Python/JetPack 조합을 우선하고, 설치가 맞지 않으면 TensorFlow 컨테이너나 ONNX 변환 경로를 별도 검증한다.

In [6]:
def get_tflite_interpreter(model_path):
    try:
        from tflite_runtime.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter
    interpreter = Interpreter(model_path=str(model_path), num_threads=max(1, (os.cpu_count() or 2)//2))
    interpreter.allocate_tensors()
    return interpreter

def read_wav_mono(path, target_sr=16000):
    import librosa
    wav, _ = librosa.load(path, sr=target_sr, mono=True)
    return wav.astype(np.float32)

def yamnet_predict(interpreter, waveform):
    inp = interpreter.get_input_details()[0]
    needed = int(np.prod(inp['shape']))
    if len(waveform) < needed: waveform = np.pad(waveform, (0, needed-len(waveform)))
    if len(waveform) > needed:
        start = (len(waveform)-needed)//2
        waveform = waveform[start:start+needed]
    x = waveform.reshape(inp['shape']).astype(inp['dtype'])
    interpreter.set_tensor(inp['index'], x)
    t0 = time.perf_counter(); interpreter.invoke(); ms = (time.perf_counter()-t0)*1000
    outputs = [interpreter.get_tensor(o['index']) for o in interpreter.get_output_details()]
    scores = next(o for o in outputs if o.shape[-1] == 521)
    return scores.squeeze(), ms

try:
    yamnet = get_tflite_interpreter(MODELS / 'yamnet.tflite')
    input_shape = yamnet.get_input_details()[0]['shape'].tolist()
    for _ in range(3): yamnet_predict(yamnet, np.zeros(int(np.prod(input_shape)), np.float32))
    times = [yamnet_predict(yamnet, np.zeros(int(np.prod(input_shape)), np.float32))[1] for _ in range(20)]
    yamnet_runtime = {'input_shape': input_shape, 'median_ms': round(statistics.median(times), 2), 'p95_ms': round(float(np.percentile(times, 95)), 2)}
except Exception as exc:
    yamnet_runtime = {'status': 'SKIPPED', 'reason': repr(exc)}
yamnet_runtime

C:\Users\SSAFY\miniforge3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


C:\Users\SSAFY\cksgud\higher\S15P11E201\.benchmark-venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


{'input_shape': [15600], 'median_ms': 4.41, 'p95_ms': 5.56}

## 7. EfficientAT MN04/MN10 비교

EfficientAT 공식 구현은 32 kHz mono → 128-bin mel spectrogram 전처리를 사용한다. 다음 셀은 공식 저장소를 cache에 clone하고 모델 로딩·전처리 포함 추론을 준비한다. 체크포인트 크기와 공식 mAP 비교는 바로 가능하지만, 웃음 F1 비교는 동일 WAV 평가 폴더가 있을 때만 의미가 있다.

In [7]:
audio_candidates = pd.DataFrame([
    {'model': 'YAMNet TFLite', 'file_MiB': (MODELS/'yamnet.tflite').stat().st_size/1024**2, 'AudioSet_mAP': np.nan, 'runtime': 'TFLite', 'sample_rate': 16000},
    {'model': 'EfficientAT MN04', 'file_MiB': (MODELS/'efficientat_mn04_as.pt').stat().st_size/1024**2, 'AudioSet_mAP': 0.432, 'runtime': 'PyTorch', 'sample_rate': 32000},
    {'model': 'EfficientAT MN10', 'file_MiB': (MODELS/'efficientat_mn10_as.pt').stat().st_size/1024**2, 'AudioSet_mAP': 0.471, 'runtime': 'PyTorch', 'sample_rate': 32000},
])
audio_candidates.round({'file_MiB': 2})

,model,file_MiB,AudioSet_mAP,runtime,sample_rate
0,YAMNet TFLite,3.94,NaN,TFLite,16000
1,EfficientAT MN04,3.88,0.432,PyTorch,32000
2,EfficientAT MN10,18.80,0.471,PyTorch,32000


In [8]:
# 선택 실행: 공식 EfficientAT 구현으로 실 WAV 추론
# EAT = CACHE / 'EfficientAT'
# if not EAT.exists(): subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/fschmid56/EfficientAT.git', str(EAT)], check=True)
# sys.path.insert(0, str(EAT))
# import torch, librosa
# from models.mn.model import get_model
# from models.preprocess import AugmentMelSTFT
# model = get_model(width_mult=0.4, pretrained_name='mn04_as').eval()
# mel = AugmentMelSTFT(n_mels=128, sr=32000, win_length=800, hopsize=320).eval()
# wav, _ = librosa.load('data/eval/laughter/example.wav', sr=32000, mono=True)
# with torch.no_grad():
#     spec = mel(torch.from_numpy(wav[None, :]))
#     logits, _ = model(spec.unsqueeze(0))
#     scores = torch.sigmoid(logits).squeeze().numpy()
# 공식 helpers.utils.labels에서 'Laughter' 관련 인덱스를 찾아 점수를 집계한다.

## 8. 최종 결정 및 적용값

### 최종적으로 사용할 모델

1. **사람 탐지: YOLOv8n TensorRT FP16**
   - Jetson Orin Nano GPU에서 실행
   - `imgsz=640`, `batch=1`, `classes=[0]`으로 시작
   - 목표 추론 주기 10~15 FPS; 녹화 인코더 안정성을 우선
2. **웃음·생활음 감지: YAMNet TFLite**
   - 두 보드 구성은 Raspberry Pi 5 CPU에서 16 kHz mono 입력으로 실행
   - Jetson 단독 구성은 Jetson CPU에서 실행하여 GPU/TensorRT와 자원 분리
   - 단일 점수가 아니라 2개 이상 연속 창 또는 이동평균과 on/off 이중 임계값으로 이벤트 확정
3. **EfficientAT MN04/MN10은 현재 제품 경로에서 제외**
   - MN04는 공개 샘플 F1이 YAMNet보다 약 1.96%p 높았지만 로컬 CPU 중앙 지연이 약 4.1배였다. 교체 기준인 5%p 개선을 충족하지 못했다.
   - MN10은 크기가 크고 공개 한국어 샘플 결과도 불안정해 사용하지 않는다.

### 한 줄 결론

> **Jetson에는 YOLOv8n TensorRT FP16, Raspberry Pi 5에는 YAMNet TFLite를 사용한다. EfficientAT은 당장 사용하지 않는다.**

### 실기기에서 마지막으로 확인할 것

- Jetson 15W/25W 모드별 YOLO median·p95·FPS·온도·메모리
- H.264 녹화·라이브 송출과 YOLO 동시 실행 시 30분 드롭 프레임 여부
- Pi 5 YAMNet 스레드 1/2/4별 지연·CPU·온도
- 실제 집 환경에서 웃음 Recall·Precision·FP/hour 및 TV 웃음 오탐

실데이터에서 MN04가 YAMNet보다 F1이 5%p 이상 높고 FP/hour와 장시간 안정성까지 만족할 때만 오디오 모델 교체를 다시 검토한다.

## 9. 출처

- [Ultralytics YOLOv8 공식 문서](https://docs.ultralytics.com/models/yolov8/) — COCO mAP, 파라미터, FLOPs, 공식 속도, export 방식
- [TensorFlow YAMNet 공식 튜토리얼](https://www.tensorflow.org/hub/tutorials/yamnet) — 521 AudioSet 클래스, 16 kHz mono 입력, MobileNet v1 구조
- [YAMNet 공식 소스/모델](https://github.com/tensorflow/models/tree/master/research/audioset/yamnet)
- [EfficientAT 공식 저장소](https://github.com/fschmid56/EfficientAT) — 모델·MIT License·체크포인트
- [JKU 연구 포털의 EfficientAT 논문](https://research.jku.at/en/publications/efficient-large-scale-audio-tagging-via-transformer-to-cnn-knowle/) — 개발 연구팀·소속·ICASSP 2023 논문
- [Ultralytics 라이선스 안내](https://www.ultralytics.com/license) — AGPL-3.0과 Enterprise 조건
- [NVIDIA TensorRT 문서](https://docs.nvidia.com/tensorrt/) — Jetson 최적화 런타임
- [NVIDIA Jetson Orin 제품 정보](https://www.nvidia.com/autonomous-machines/embedded-systems/jetson-orin/) — Orin Nano 전력/AI 성능 범위
- [Deeply Nonverbal Vocalization Dataset](https://github.com/deeplyinc/Nonverbal-Vocalization-Dataset) / [OpenSLR 99](https://www.openslr.org/99/) — 한국인 비언어 발성 공개 샘플 및 라이선스

라이선스 메모: Ultralytics YOLOv8은 AGPL-3.0 또는 Enterprise 조건이다. 특히 비공개·상용·임베디드 배포는 출시 전에 적용 조건을 확인한다. YAMNet 공식 모델 카드와 데이터셋 조건도 제품 사용 전에 재확인한다.